# Fish disease classifier - MatsyaDx-BD, honest fish-level evaluation

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > **Run all** (about 15-30 min).

What this notebook does differently from a normal tutorial:
* The dataset has only **86 individual fish** (about 25 photos each, shot seconds apart). Splitting photos randomly
  would put the same fish in train and test and give fake, very high accuracy. Here every test photo comes from a
  fish the model has **never seen** (5-fold cross-validation, grouped by fish).
* At most 30 photos per fish are used, evenly spread, so a few fish with 100+ photos don't dominate.
* Pixel normalization is built INTO the model: the app feeds raw 0-255 pixel values.

**You get one download:** `fish_model_bundle.zip` = `fish_disease_model.tflite` + `labels.txt` + `cv_report.txt`.

In [2]:
# 1. Setup
import os, glob, subprocess, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report

SMOKE = os.environ.get("FISH_SMOKE") == "1"          # tiny offline self-test only; leave unset in Colab
WORK = os.environ.get("FISH_WORK", "/content")
URL = "https://data.mendeley.com/public-api/zip/sxkynv9t7n/download/2"
RAW, EXTRACTED, SMALL, OUT = f"{WORK}/matsya", f"{WORK}/matsya_x", f"{WORK}/small", f"{WORK}/out"

IMG = 128 if SMOKE else 224        # model input size - must match _imageSize in the Flutter app
MAX_PER_FISH = 30
N_FOLDS = 2 if SMOKE else 5
HEAD_EPOCHS = 1 if SMOKE else 6    # stage 1: train only the new classifier layer
FT_EPOCHS = 1 if SMOKE else 14     # stage 2: gently fine-tune the top of the pretrained network
BATCH = 32
SEED = 42
PRETRAINED = not SMOKE
CLASSES = ["bacterial_gill_disease", "bacterial_red_disease", "eus", "healthy"]
CLASS_MAP = {"Bacterial Gill Diseases": "bacterial_gill_disease",
             "Bacterial Red Diseases": "bacterial_red_disease",
             "Epizootic Ulcerative Syndrome (EUS)": "eus",
             "Healthy Fish": "healthy"}

tf.keras.utils.set_random_seed(SEED)
os.makedirs(OUT, exist_ok=True)
LOG = []
def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    LOG.append(s)

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or
      "NONE -> set Runtime > Change runtime type > T4 GPU, otherwise training is very slow")

TensorFlow 2.21.0 | GPU: NONE -> set Runtime > Change runtime type > T4 GPU, otherwise training is very slow


In [3]:
# 2. Get the data (skipped automatically if you already downloaded and extracted it)
if not os.path.exists(EXTRACTED):
    if not glob.glob(f"{RAW}/*/MatsyaDx-BD/metadata.csv"):
        subprocess.run(["wget", "-q", "-L", "-O", f"{WORK}/matsya.zip", URL], check=True)
        subprocess.run(["unzip", "-q", "-o", f"{WORK}/matsya.zip", "-d", RAW], check=True)
        os.remove(f"{WORK}/matsya.zip")
    subprocess.run("apt-get -qq install -y p7zip-full", shell=True)
    meta_dir = os.path.dirname(glob.glob(f"{RAW}/*/MatsyaDx-BD/metadata.csv")[0])
    os.makedirs(EXTRACTED, exist_ok=True)
    for a in sorted(glob.glob(meta_dir + "/*.7z")):
        print("extracting", os.path.basename(a), flush=True)
        subprocess.run(["7z", "x", "-y", a, f"-o{EXTRACTED}"], check=True, stdout=subprocess.DEVNULL)
META = glob.glob(f"{RAW}/*/MatsyaDx-BD/metadata.csv")[0]

KeyboardInterrupt: 

In [ ]:
# 3. Read metadata, group photos by individual fish, cap photos per fish
df = pd.read_csv(META)
df["path"] = EXTRACTED + "/" + df["image_path"]
df["label"] = df["health_condition"].map(CLASS_MAP)
df["y"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})
assert df["y"].notna().all(), "unexpected class name in metadata.csv"
# 'Fish_1' exists in every class and species, so the real fish identity is class + species + specimen id
df["group"] = df["health_condition"] + "|" + df["fish_category"] + "|" + df["specimen_id"]
df["date"] = df["image_id"].str.extract(r"IMG(\d{8})")[0]
n_missing = int((~df["path"].map(os.path.exists)).sum())
assert n_missing == 0, f"{n_missing} image files from metadata.csv are missing - extraction incomplete?"

def pick_even(g, k):
    g = g.sort_values("image_id")                      # filenames are timestamps
    if len(g) <= k:
        return g
    return g.iloc[np.linspace(0, len(g) - 1, k).round().astype(int)]

n_all = len(df)
df = pd.concat([pick_even(g, MAX_PER_FISH) for _, g in df.groupby("group")]).reset_index(drop=True)
log(f"{n_all} photos -> {len(df)} used (max {MAX_PER_FISH} per fish, evenly spaced) from {df['group'].nunique()} fish")
log(df.groupby("label").agg(photos=("path", "size"), fish=("group", "nunique")).to_string())

2137 photos -> 1511 used (max 30 per fish, evenly spaced) from 86 fish
                        photos  fish
label                               
bacterial_gill_disease     275    27
bacterial_red_disease      634    35
eus                        302    14
healthy                    300    10


In [ ]:
# 4. Make small copies (the originals are 4000x3000 - far too big to train on directly)
os.makedirs(SMALL, exist_ok=True)
df["small"] = [f"{SMALL}/{i:05d}.jpg" for i in range(len(df))]

def shrink(args):
    src, dst = args
    if os.path.exists(dst):
        return
    im = Image.open(src)
    im.draft("RGB", (1000, 1000))                      # fast JPEG decode at reduced size
    im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((448, 448), Image.LANCZOS)
    im.save(dst, quality=95)

with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
    list(ex.map(shrink, zip(df["path"], df["small"])))
print("small copies ready:", len(df))

small copies ready: 1511


In [ ]:
# 5. Input pipeline + model
def load(path, y):
    x = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    x = tf.image.resize(x, (IMG, IMG), method="bilinear", antialias=True)   # squashed to a square, as in the app
    return x, y

def augment(x, y):
    x = tf.image.random_flip_left_right(x)
    size = tf.cast(tf.round(IMG * tf.random.uniform([], 1.0, 1.25)), tf.int32)   # random zoom + shift
    x = tf.image.resize(x, (size, size))
    x = tf.image.random_crop(x, (IMG, IMG, 3))
    x = tf.image.random_brightness(x, 25.0)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    x = tf.image.random_saturation(x, 0.8, 1.2)
    x = tf.ensure_shape(tf.clip_by_value(x, 0.0, 255.0), (IMG, IMG, 3))
    return x, y

def make_ds(d, train):
    ds = tf.data.Dataset.from_tensor_slices((d["small"].values, d["y"].values.astype("int32")))
    if train:
        ds = ds.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load, tf.data.AUTOTUNE)
    if train:
        ds = ds.map(augment, tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

def build_model():
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False,
                                             weights="imagenet" if PRETRAINED else None)
    inp = tf.keras.Input((IMG, IMG, 3), name="input")             # raw pixel values 0..255
    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(inp)      # MobileNetV2 wants [-1, 1]: done inside the model
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(len(CLASSES), name="output")(x)   # logits (the app applies softmax)
    return tf.keras.Model(inp, out), base

def compile_(m, lr):
    m.compile(tf.keras.optimizers.Adam(lr),
              tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])

def train_model(train_df):
    counts = np.bincount(train_df["y"], minlength=len(CLASSES))
    cw = {i: len(train_df) / (len(CLASSES) * max(int(c), 1)) for i, c in enumerate(counts)}   # rarer class = bigger weight
    tr = make_ds(train_df, True)
    model, base = build_model()
    base.trainable = False
    compile_(model, 1e-3)
    model.fit(tr, epochs=HEAD_EPOCHS, class_weight=cw, verbose=2)
    base.trainable = True
    for layer in base.layers[:-50]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_(model, 1e-4)
    model.fit(tr, epochs=FT_EPOCHS, class_weight=cw, verbose=2)
    return model            # fixed schedule, nothing is tuned on the test fish

def predict_probs(model, d):
    z = model.predict(make_ds(d, False), verbose=0)
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

In [ ]:
# 6. Cross-validation: every photo is predicted by a model that never saw that fish
oof = np.zeros((len(df), len(CLASSES)))
skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr_i, va_i) in enumerate(skf.split(df, df["y"], df["group"]), 1):
    tr_d, va_d = df.iloc[tr_i], df.iloc[va_i]
    assert not set(tr_d["group"]) & set(va_d["group"]), "a fish is in both train and test!"
    log(f"\n=== fold {k}/{N_FOLDS}: train {len(tr_d)} photos / {tr_d['group'].nunique()} fish"
        f" | test {len(va_d)} photos / {va_d['group'].nunique()} fish ===")
    m = train_model(tr_d)
    oof[va_i] = predict_probs(m, va_d)
    log(f"fold {k} photo accuracy: {(oof[va_i].argmax(1) == va_d['y'].values).mean():.3f}")
    del m
    tf.keras.backend.clear_session()

NameError: name 'np' is not defined

In [ ]:
# 7. Honest results
H = CLASSES.index("healthy")
df["pred"], df["conf"] = oof.argmax(1), oof.max(1)
df["ok"] = df["pred"] == df["y"]
log("\n##### RESULTS - every photo predicted by a model that never saw that fish #####")
log(f"Photo-level accuracy: {df['ok'].mean():.3f}")
log(classification_report(df["y"], df["pred"], labels=list(range(len(CLASSES))), target_names=CLASSES, digits=3, zero_division=0))

gp = pd.DataFrame(oof, index=df["group"].values).groupby(level=0).mean()      # average each fish's photos
gy = df.groupby("group")["y"].first().reindex(gp.index)
gpred = gp.values.argmax(1)
log(f"Fish-level accuracy (photos of one fish averaged): {(gpred == gy.values).mean():.3f} over {len(gy)} fish")
log("(only a few fish per class, so treat every number here as +/- 10 points)")
log(pd.crosstab(pd.Series(np.array(CLASSES)[gy.values], name="true"),
                pd.Series(np.array(CLASSES)[gpred], name="predicted")).to_string())

sick = df["y"] != H
log(f"\nDiseased photos wrongly called HEALTHY: {(df['pred'][sick] == H).mean():.1%}   <- the dangerous mistake")
log(f"Healthy photos wrongly called diseased: {(df['pred'][~sick] != H).mean():.1%}")
log("\nPhoto accuracy by class x species:\n" +
    df.pivot_table(index="label", columns="fish_category", values="ok", aggfunc="mean").round(2).to_string())
log("\nPhoto accuracy by class x capture date (big gaps between dates = the model may be reading the day/background):\n" +
    df.pivot_table(index="label", columns="date", values="ok", aggfunc="mean").round(2).to_string())
log("\nIf the app only answered when confident:")
for t in (0.5, 0.6, 0.7, 0.8, 0.9):
    mk = df["conf"] >= t
    log(f"  confidence >= {t:.1f}: answers {mk.mean():.0%} of photos, accuracy {df['ok'][mk].mean():.3f}" if mk.any()
        else f"  confidence >= {t:.1f}: no photos")

In [ ]:
# 8. Train the final model on ALL fish and export it for the app
log("\n##### Final model (trained on all fish) #####")
final = train_model(df)
try:
    tflite = tf.lite.TFLiteConverter.from_keras_model(final).convert()
except Exception as e:
    log("from_keras_model failed:", repr(e)[:200], "-> trying SavedModel route")
    final.export(f"{WORK}/saved_model")
    tflite = tf.lite.TFLiteConverter.from_saved_model(f"{WORK}/saved_model").convert()
open(f"{OUT}/fish_disease_model.tflite", "wb").write(tflite)
open(f"{OUT}/labels.txt", "w").write("\n".join(CLASSES) + "\n")

try:
    it = tf.lite.Interpreter(model_content=tflite)
except Exception:                      # newer TensorFlow versions removed it -> use the standalone LiteRT package
    subprocess.run("pip install -q ai-edge-litert", shell=True)
    from ai_edge_litert.interpreter import Interpreter
    it = Interpreter(model_content=tflite)
it.allocate_tensors()
i_in, i_out = it.get_input_details()[0], it.get_output_details()[0]
log("TFLite input:", [int(v) for v in i_in["shape"]], "| output:", [int(v) for v in i_out["shape"]])

def tfl_logits(x):
    it.set_tensor(i_in["index"], x.astype("float32")[None])
    it.invoke()
    return it.get_tensor(i_out["index"])[0].copy()

sample = df.sample(min(30, len(df)), random_state=1)
keras_z = final.predict(make_ds(sample, False), verbose=0)
tfl_z = np.array([tfl_logits(load(p, 0)[0].numpy()) for p in sample["small"]])
log(f"TFLite vs Keras: max logit difference {np.abs(tfl_z - keras_z).max():.5f}, "
    f"same answer on {(tfl_z.argmax(1) == keras_z.argmax(1)).mean():.0%} of photos")

In [ ]:
# 9. How fragile is the model to the app's resize method? (full-size photos, as image_picker delivers them)
def full_photo_input(path, method):
    im = Image.open(path)
    im.draft("RGB", (2000, 2000))
    im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((1024, 1024), Image.BILINEAR)                    # image_picker maxWidth: 1024
    return np.asarray(im.resize((IMG, IMG), method), dtype=np.float32)

rs = sample.head(20)
ref = np.array([tfl_logits(full_photo_input(p, Image.BILINEAR)).argmax() for p in rs["path"]])
for name, meth in [("nearest-neighbour (the OLD app resize)", Image.NEAREST), ("box/average", Image.BOX)]:
    got = np.array([tfl_logits(full_photo_input(p, meth)).argmax() for p in rs["path"]])
    log(f"{name}: same answer as the training-style resize on {(got == ref).mean():.0%} of {len(rs)} full-size photos")

In [ ]:
# 10. Download everything as ONE zip
open(f"{OUT}/cv_report.txt", "w").write("\n".join(LOG))
shutil.make_archive(f"{WORK}/fish_model_bundle", "zip", OUT)
print("model size: %.1f MB" % (os.path.getsize(f"{OUT}/fish_disease_model.tflite") / 1e6))
try:
    from google.colab import files
    files.download(f"{WORK}/fish_model_bundle.zip")
except ImportError:
    print("saved:", f"{WORK}/fish_model_bundle.zip")

### After the download
Send the contents of `cv_report.txt` (or paste the "RESULTS" output above) back to the chat.
The Flutter app then needs: `_imageSize = 224`, raw 0-255 pixel input (no mean/std - it's inside the model),
4 labels (a new **EUS** entry), and re-tuned confidence thresholds. Those changes depend on your results.